# SEED-VII EEGNet Encoder Pipeline

**重构版 (2026-05-27)：EEGNet 主力，无 ICA，per-subject .npz 预处理**  
**更新 (2026-05-28)：新增 `per_subject` 单被试 trial-level 分割模式**

## 执行流程
1. **Cell 0**: 安装依赖
2. **Cell 1**: 环境配置 & 路径设置
3. **Cell 2**: 预处理 — 20 个 .mat → 20 个 .npz
4. **Cell 3**: 上传 .npz 到 ModelScope
5. **Cell 4**: (可选) 从 ModelScope 下载 .npz
6. **Cell 5**: 训练 — **全被试混合 trial-level 分割**（原始行为）
7. **Cell 6**: 训练 — ★ **单被试 trial-level 分割**（新功能）
8. **Cell 7**: 续训（支持两种模式）
9. **Cell 8**: 编码器推理
10. **Cell 9**: 查看训练日志 & 对比两种分割结果

---

## 两种分割模式说明

| 模式 | 参数 | 场景 | 说明 |
|------|------|------|------|
| 全被试混合分割 | `--split-mode all` | 跨被试泛化 | 所有被试的 trial 混合后随机分割，val/test 包含各被试的 trial（原始行为）|
| 单被试独立分割 | `--split-mode per_subject` | 被试内泛化 | 每个被试在自己的 trial 内单独分割，再合并，每个被试都在 train/val/test 中有代表 |

两种方式均 **严格避免数据泄漏**：同一 trial 的所有时间窗口只出现在一个 split。

---

## Cell 0: 安装依赖

In [ ]:
!pip install -q numpy scipy torch tqdm h5py pyyaml modelscope

## Cell 1: 环境配置 & 路径设置

**请根据你的实际环境修改以下路径变量！**

In [ ]:
import os, sys
from pathlib import Path

# ========== 路径配置 ==========
# 项目根目录（包含 src/, scripts/ 等）
PROJECT_ROOT = Path("/mnt/workspace/EEG_encoder_version2")  # 修改为你的实际路径

# .mat 数据目录（20 个 {1..20}.mat 文件）
MAT_DIR = Path("/mnt/workspace/data/EEG_preprocessed")     # 修改为你的实际路径

# 连续标签 CSV 目录（可选）
SAVE_INFO_DIR = Path("/mnt/workspace/data/save_info")       # 没有则留空字符串

# 预处理输出目录（存放 20 个 .npz 文件）
NPZ_DIR = Path("/mnt/workspace/preprocessed")

# 训练输出目录（全被试混合分割）
RUNS_DIR_ALL = Path("/mnt/workspace/runs_all")

# 训练输出目录（单被试独立分割）
RUNS_DIR_PER_SUBJ = Path("/mnt/workspace/runs_per_subject")

# 编码输出
ENCODED_OUTPUT_ALL      = Path("/mnt/workspace/encoded_all.npz")
ENCODED_OUTPUT_PER_SUBJ = Path("/mnt/workspace/encoded_per_subject.npz")

# ModelScope 配置
MODELSCOPE_DATASET = "DEREKVERSE/SEED-VII"
NPZ_REPO_PREFIX    = "preprocessed_npz"

# ========== Token ==========
# 设置 ModelScope token（从 https://modelscope.cn/my/myaccesstoken 获取）
# os.environ["MODELSCOPE_API_TOKEN"] = "your_token_here"

# ========== 确保 src/ 在 Python path 中 ==========
sys.path.insert(0, str(PROJECT_ROOT))

# 验证
print(f"PROJECT_ROOT: {PROJECT_ROOT} (exists: {PROJECT_ROOT.exists()})")
print(f"MAT_DIR:      {MAT_DIR} (exists: {MAT_DIR.exists()})")
print(f"NPZ_DIR:      {NPZ_DIR}")
print(f"RUNS_DIR_ALL:      {RUNS_DIR_ALL}")
print(f"RUNS_DIR_PER_SUBJ: {RUNS_DIR_PER_SUBJ}")
print(f"Token set: {'MODELSCOPE_API_TOKEN' in os.environ}")

## Cell 2: 预处理 — 20 个 .mat → 20 个 per-subject .npz

流程（Design.md，跳过 ICA）：
```
原始 EEG (62通道, 200Hz, 已带通+陷波)
    ↓ 基线校正（减均值）
    ↓ 平均参考 CAR
    ↓ 居中 60% 裁剪
    ↓ 4秒窗口 50%重叠（每 clip 最多 60 个居中窗口）
    ↓ 按通道 z-score
    → {subject}.npz
```

In [ ]:
!python {PROJECT_ROOT}/scripts/preprocess_to_npz.py \
  --mat-dir {MAT_DIR} \
  --output-dir {NPZ_DIR} \
  --save-info-dir {SAVE_INFO_DIR if SAVE_INFO_DIR.exists() else ''} \
  --window-seconds 4.0 \
  --step-seconds 2.0 \
  --middle-ratio 0.6 \
  --max-windows-per-trial 60 \
  --compress

In [ ]:
# 验证预处理结果
import numpy as np

for p in sorted(NPZ_DIR.glob("*.npz")):
    data = np.load(p, allow_pickle=True)
    print(f"{p.name}: X={data['X'].shape}, y={data['y'].shape}, "
          f"labels={dict(zip(*np.unique(data['y'], return_counts=True)))}")

## Cell 3: 上传 .npz 到 ModelScope

将预处理好的 20 个 .npz 文件回写到 ModelScope 数据集。

In [ ]:
!python {PROJECT_ROOT}/scripts/upload_npz_to_modelscope.py \
  --npz-dir {NPZ_DIR} \
  --dataset {MODELSCOPE_DATASET} \
  --path-prefix {NPZ_REPO_PREFIX}

## Cell 4: (可选) 从 ModelScope 下载 .npz

如果在另一台机器上训练，先拉取预处理好的 .npz 文件。

In [ ]:
# 只在需要时取消注释运行
# !python {PROJECT_ROOT}/scripts/download_npz_from_modelscope.py \
#   --dataset {MODELSCOPE_DATASET} \
#   --path-prefix {NPZ_REPO_PREFIX} \
#   --local-dir {NPZ_DIR}

## Cell 5: 训练 — 全被试混合 trial-level 分割（`--split-mode all`，原始行为）

**策略**：所有 20 个被试的 trial 混合后随机做 trial-level 分割。  
验证/测试集中每个被试都有 trial 留出，适合评估**跨被试泛化**能力。

```
全部 trial (Subject 1~20 混合)
    ↓ 随机 shuffle
    ├── train trials (80%)
    ├── val trials   (10%)
    └── test trials  (10%)
```

训练策略：
1. 前 10 epoch 只开分类损失 `L_cls` 预训练
2. 之后联合训练 `L_cls + L_reg`（退化方案，`L_rank` 暂不开启）
3. 余弦退火学习率，最小 1e-5
4. 10 小时软超时，优雅保存

In [ ]:
!python {PROJECT_ROOT}/scripts/train.py \
  --data-dir {NPZ_DIR} \
  --output-dir {RUNS_DIR_ALL} \
  --split-mode all \
  --model-type eegnet \
  --device auto \
  --amp \
  --batch-size 256 \
  --lr 2e-4 \
  --pretrain-epochs 10 \
  --max-epochs 200 \
  --patience 30 \
  --max-runtime-hours 10

## Cell 6: ★ 训练 — 单被试 trial-level 分割（`--split-mode per_subject`，新功能）

**策略**：对每个被试**独立**做 trial-level 分割，再合并成全局 train/val/test。  
每个被试在 train/val/test 中均有代表，适合评估**被试内泛化**能力。

```
Subject 1 trials                  Subject 2 trials     ...  Subject 20 trials
    ├── train (80%)   ─┐               ├── train ─┐              ├── train ─┐
    ├── val   (10%)   ─┼─► 合并        ├── val   ─┼─► 合并        ├── val   ─┼─► 合并
    └── test  (10%)   ─┘               └── test  ─┘              └── test  ─┘
                        ↓                          ↓                          ↓
                    全局 train             全局 val              全局 test
```

⚠️ **数据泄漏防护**：同一 trial 的所有时间窗口只属于一个 split，绝不跨 split 出现。

**适用场景**：
- 想让模型在每个被试内的 held-out trial 上验证表现
- 训练一个对所有被试均有覆盖的「被试内编码器」
- 对比同一被试在不同情绪下的 EEG 表示质量

In [ ]:
!python {PROJECT_ROOT}/scripts/train.py \
  --data-dir {NPZ_DIR} \
  --output-dir {RUNS_DIR_PER_SUBJ} \
  --split-mode per_subject \
  --model-type eegnet \
  --device auto \
  --amp \
  --batch-size 256 \
  --lr 2e-4 \
  --pretrain-epochs 10 \
  --max-epochs 200 \
  --patience 30 \
  --max-runtime-hours 10

In [ ]:
# ★ 在 Python 中直接调用 split_trials_per_subject 预览分割结果（无需运行完整训练）
import json
import numpy as np
from src.dataset import scan_npz_metadata, split_trials_per_subject, split_trials_from_meta

# 轻量扫描（只读 y/s/meta）
npz_paths, y_all, s_all, meta, file_map = scan_npz_metadata(str(NPZ_DIR))

print("\n" + "="*60)
print("单被试 trial-level 分割 (per_subject):")
print("="*60)
splits_ps = split_trials_per_subject(meta, val_ratio=0.1, test_ratio=0.1, seed=42)
print(f"\n  汇总: train={len(splits_ps['train'])}, "
      f"val={len(splits_ps['val'])}, test={len(splits_ps['test'])} windows")

print("\n" + "="*60)
print("全被试混合 trial-level 分割 (all):")
print("="*60)
splits_all = split_trials_from_meta(meta, val_ratio=0.1, test_ratio=0.1, seed=42)
print(f"  汇总: train={len(splits_all['train'])}, "
      f"val={len(splits_all['val'])}, test={len(splits_all['test'])} windows")

# 验证无数据泄漏：检查 train/val/test 索引无交集
def check_no_leak(splits, name):
    tr = set(splits['train'].tolist())
    va = set(splits['val'].tolist())
    te = set(splits['test'].tolist())
    tr_va = tr & va
    tr_te = tr & te
    va_te = va & te
    leak = len(tr_va) + len(tr_te) + len(va_te)
    print(f"\n[{name}] 数据泄漏检查: "
          f"train∩val={len(tr_va)}, train∩test={len(tr_te)}, val∩test={len(va_te)} "
          f"→ {'✅ 无泄漏' if leak == 0 else '❌ 有泄漏！'}")

check_no_leak(splits_ps, "per_subject")
check_no_leak(splits_all, "all")

## Cell 7: 续训 (Resume)

如果训练被中断（超时或其他原因），使用 `--resume` 从断点继续。  
**注意**：续训时 `--split-mode` 需与原始训练保持一致。

In [ ]:
# 续训 — 全被试混合分割模式
# !python {PROJECT_ROOT}/scripts/train.py \
#   --data-dir {NPZ_DIR} \
#   --output-dir {RUNS_DIR_ALL} \
#   --split-mode all \
#   --model-type eegnet \
#   --device auto --amp \
#   --resume \
#   --max-runtime-hours 10

# 续训 — 单被试分割模式
# !python {PROJECT_ROOT}/scripts/train.py \
#   --data-dir {NPZ_DIR} \
#   --output-dir {RUNS_DIR_PER_SUBJ} \
#   --split-mode per_subject \
#   --model-type eegnet \
#   --device auto --amp \
#   --resume \
#   --max-runtime-hours 10

print("取消注释上方代码块以执行续训。")

## Cell 8: 编码器推理

用训练好的模型导出 embedding + 预测结果。可分别对两种模型进行推理。

In [ ]:
# 推理 — 全被试混合分割训练的模型
!python {PROJECT_ROOT}/scripts/encode.py \
  --data-dir {NPZ_DIR} \
  --checkpoint {RUNS_DIR_ALL}/best_encoder.pt \
  --output {ENCODED_OUTPUT_ALL} \
  --model-type eegnet \
  --device auto

In [ ]:
# 推理 — 单被试分割训练的模型
!python {PROJECT_ROOT}/scripts/encode.py \
  --data-dir {NPZ_DIR} \
  --checkpoint {RUNS_DIR_PER_SUBJ}/best_encoder.pt \
  --output {ENCODED_OUTPUT_PER_SUBJ} \
  --model-type eegnet \
  --device auto

In [ ]:
# 查看编码结果
import numpy as np

for label, path in [("全被试分割", ENCODED_OUTPUT_ALL),
                    ("单被试分割", ENCODED_OUTPUT_PER_SUBJ)]:
    if not path.exists():
        print(f"[{label}] 文件不存在: {path}")
        continue
    data = np.load(str(path), allow_pickle=True)
    acc  = (data['cls_pred'] == data['labels']).mean()
    print(f"[{label}]")
    print(f"  Features:       {data['features'].shape}")
    print(f"  Class preds:    {data['cls_pred'].shape}")
    print(f"  Intensity pred: {data['intensity_pred'].shape}")
    print(f"  Labels:         {data['labels'].shape}")
    print(f"  Overall acc:    {acc:.4f}")
    print()

## Cell 9: 查看训练日志 & 对比两种分割结果

In [ ]:
import json

print("="*60)
print("训练摘要对比")
print("="*60)

for label, runs_dir in [("全被试混合分割 (all)", RUNS_DIR_ALL),
                        ("单被试独立分割 (per_subject)", RUNS_DIR_PER_SUBJ)]:
    summary_path = runs_dir / "summary.json"
    print(f"\n[{label}]")
    if summary_path.exists():
        with open(summary_path) as f:
            summary = json.load(f)
        print(json.dumps(summary, indent=2, ensure_ascii=False))
    else:
        print(f"  No summary.json found at {summary_path}")

print("\n" + "="*60)
print("最后 20 行训练日志")
print("="*60)

for label, runs_dir in [("all", RUNS_DIR_ALL),
                        ("per_subject", RUNS_DIR_PER_SUBJ)]:
    log_path = runs_dir / "train.log"
    if log_path.exists():
        with open(log_path) as f:
            lines = f.readlines()
        print(f"\n--- [{label}] Last 20 lines ---")
        for line in lines[-20:]:
            print(line, end="")
    else:
        print(f"  [{label}] No train.log found.")